In [ ]:
import pandas as pd
import re
from pathlib import Path
import os
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd

description_files = [
    "AML_msigdb_descriptions.csv",
    "Lung_msigdb_descriptions.csv",
    "BreastCancer msigdb_description.csv"
]

# Validation files
validation_files = [
    "AML Validation V1.csv",
    "Breast Cancer Validation V1.csv",
    "Lung Cancer Validation V1.csv"
]

descriptions_df = pd.concat(
    [pd.read_csv(f) for f in description_files],
    ignore_index=True
)

validations_df = pd.concat(
    [pd.read_csv(f) for f in validation_files],
    ignore_index=True
)

descriptions_df.to_csv("Combined_msigdb_descriptions.csv", index=False)
validations_df.to_csv("Combined_Validation_V1.csv", index=False)

print(f"Descriptions: {len(descriptions_df)} rows")
print(f"Validations: {len(validations_df)} rows")

print("Saved:")
print(" - Combined_msigdb_descriptions.csv")
print(" - Combined_Validation_V1.csv")

In [ ]:
import pandas as pd

input_file = "Combined_Validation_V1.csv"
output_file = "Combined_annotation_ablation.csv"

df = pd.read_csv(input_file)


df["Final_Without_Validation"] = df.apply(
    lambda row: row["Process_With_Enrichment_Original"]
    if row["Confidence_With_Enrichment_Before"] >= row["Confidence_Without_Enrichment_Before"]
    else row["Process_Without_Enrichment_Original"],
    axis=1
)

new_df = pd.DataFrame({
    "Genes": df["Genes"],
    "Geneset": df["GeneSet_Name"],
    "Enrichment_driven_LLM_without_validation": df["Process_With_Enrichment_Original"],
    "Direct_LLM_without_validation": df["Process_Without_Enrichment_Original"],
    "Final_without_validation": df["Final_Without_Validation"],
    "Final_with_validation": df["Final_Process"]
})


new_df.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Rows:", len(new_df))

In [ ]:
def run_ladder_validation(df,
                          MODELS=None,
                          COLORS=None,
                          METHOD_NAMES=None,
                          out_dir="ladder_validation_outputs",
                          csv_name="validation_results_general_only.csv",
                          winfig_name="Figure_WinCounts_General.png",
                          simfig_name="Figure_Similarity_General.png",
                          max_length=512,
                          device=None,
                          run_name: str = None,
                          add_timestamp: bool = False):

    prefix = f"{run_name}_" if run_name else ""
    if add_timestamp:
        from datetime import datetime
        ts = datetime.now().strftime("%Y%m%dT%H%M%S")
        prefix = f"{prefix}{ts}_" if prefix else f"{ts}_"

    if MODELS is None:
        MODELS = {
            'BioLORD-2023': 'FremyCompany/BioLORD-2023',
            'MedCPT': 'ncbi/MedCPT-Query-Encoder'
        }

    if COLORS is None:
        COLORS = {
            'Enrichment': '#1f77b4',
            'Direct': '#ff7f0e',
            'Final_no_val': '#2ca02c',
            'Final_val': '#d62728'
        }

    if METHOD_NAMES is None:
        METHOD_NAMES = {
            'Enrichment': 'Enrichment',
            'Direct': 'Direct',
            'Final_no_val': 'Final_no_val',
            'Final_val': 'Final_val'
        }

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"Using device: {device}\n")

    os.makedirs(out_dir, exist_ok=True)

    csv_final = os.path.join(out_dir, f"{prefix}{csv_name}" if prefix else csv_name)
    winfig_final = os.path.join(out_dir, f"{prefix}{winfig_name}" if prefix else winfig_name)
    simfig_final = os.path.join(out_dir, f"{prefix}{simfig_name}" if prefix else simfig_name)


    def load_model(model_id):
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModel.from_pretrained(model_id)
        model.to(device)
        model.eval()
        return tokenizer, model


    def get_embedding(text, tokenizer, model, max_length=max_length):

        if not isinstance(text, str) or len(text.strip()) == 0:
            return np.zeros(model.config.hidden_size, dtype=float)

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            emb = outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()

        return emb


    def validate_with_model(df_local, display_name, model_id):

        print(f"Validating: {display_name}")

        tokenizer, model = load_model(model_id)

        results = []

        for idx, row in tqdm(df_local.iterrows(),
                             total=len(df_local),
                             desc=f"{display_name}",
                             leave=False):

            desc_emb = get_embedding(
                str(row.get('MSigDB_Brief_Description', '')),
                tokenizer,
                model
            )

            enrich_emb = get_embedding(
                str(row.get('Enrichment_driven_LLM_without_validation', '')),
                tokenizer,
                model
            )

            direct_emb = get_embedding(
                str(row.get('Direct_LLM_without_validation', '')),
                tokenizer,
                model
            )

            final_no_val_emb = get_embedding(
                str(row.get('Final_without_validation', '')),
                tokenizer,
                model
            )

            final_val_emb = get_embedding(
                str(row.get('Final_with_validation', '')),
                tokenizer,
                model
            )

            eps = 1e-12

            enrich_sim = cosine_similarity(
                desc_emb.reshape(1, -1) + eps,
                enrich_emb.reshape(1, -1) + eps
            )[0][0]

            direct_sim = cosine_similarity(
                desc_emb.reshape(1, -1) + eps,
                direct_emb.reshape(1, -1) + eps
            )[0][0]

            final_no_val_sim = cosine_similarity(
                desc_emb.reshape(1, -1) + eps,
                final_no_val_emb.reshape(1, -1) + eps
            )[0][0]

            final_val_sim = cosine_similarity(
                desc_emb.reshape(1, -1) + eps,
                final_val_emb.reshape(1, -1) + eps
            )[0][0]


            scores = {
                'Enrichment': enrich_sim,
                'Direct': direct_sim,
                'Final_no_val': final_no_val_sim,
                'Final_val': final_val_sim
            }

            max_score = max(scores.values())

            winners = [k for k, v in scores.items()
                       if abs(v - max_score) < 1e-8]

            winner_str = "|".join([METHOD_NAMES[w] for w in winners])


            results.append({
                'Geneset': row.get('Geneset'),
                'Enrichment_Similarity': enrich_sim,
                'Direct_Similarity': direct_sim,
                'Final_no_val_Similarity': final_no_val_sim,
                'Final_val_Similarity': final_val_sim,
                'Winner': winner_str,
                'Model': display_name
            })


        del model, tokenizer

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return pd.DataFrame(results)



    all_results = []

    for model_name, model_path in MODELS.items():

        try:

            model_results = validate_with_model(
                df,
                model_name,
                model_path
            )

            all_results.append(model_results)

        except Exception as e:

            print(f"Warning: failed on model {model_name}: {e}")

            continue


    results_df = pd.concat(all_results, ignore_index=True)

    results_df.to_csv(csv_final, index=False)

    print(f"✓ Saved: {csv_final}\n")


    plt.style.use("seaborn-v0_8-whitegrid")

    plt.rcParams.update({
        "figure.dpi": 150,
        "font.family": "Arial",
        "axes.titlesize": 13,
        "axes.titleweight": "bold",
        "axes.labelsize": 11
    })


    fig, ax = plt.subplots(figsize=(10, 5))

    methods = [
        METHOD_NAMES['Enrichment'],
        METHOD_NAMES['Direct'],
        METHOD_NAMES['Final_no_val'],
        METHOD_NAMES['Final_val']
    ]

    models_list = sorted(results_df['Model'].unique())

    x = np.arange(len(models_list))

    width = 0.18


    for i, method in enumerate(methods):

        wins = []

        for model in models_list:

            model_subset = results_df[
                results_df['Model'] == model
            ]

            win_count = len(
                model_subset[
                    model_subset['Winner'].str.contains(method, regex=False)
                ]
            )

            wins.append(win_count)


        offset = (i - 1.5) * width

        bars = ax.bar(
            x + offset,
            wins,
            width,
            label=method,
            color=COLORS.get(method, None),
            alpha=0.9,
            edgecolor='black',
            linewidth=0.7
        )

        for bar, win in zip(bars, wins):

            if win > 0:

                ax.text(
                    bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 1,
                    f'{win}',
                    ha='center',
                    va='bottom',
                    fontsize=9,
                    fontweight='bold'
                )


    ax.set_xticks(x)
    ax.set_xticklabels(models_list, rotation=30, ha='right')

    ax.set_ylabel('Number of Wins')
    ax.set_title('Annotation Accuracy Across Embedding Models')

    ax.legend(frameon=True)

    sns.despine()

    plt.tight_layout()

    plt.savefig(
        winfig_final,
        dpi=600,
        bbox_inches='tight',
        facecolor='white'
    )

    plt.show()


    print("✓ Figures saved")

    return results_df

In [ ]:
combined_path = "Combined_annotation_ablation.csv"
combined = pd.read_csv(combined_path)
annotations_df = combined
msigdb_df = pd.read_csv('Combined_msigdb_descriptions.csv')
df = annotations_df.merge(msigdb_df, on='Geneset')
print(f"Total genesets: {len(df)}\n")

results_df = run_ladder_validation(df, run_name="Ablation", add_timestamp=False)